In [312]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import math

In [313]:
batch_size = 2
seq_len = 5
d_model = 512

x = torch.rand(batch_size, seq_len, d_model)  # (batch_size, seq_len, d_model)

print("Input shape:", x.shape)

Input shape: torch.Size([2, 5, 512])


In [314]:
W_q = nn.Linear(d_model, d_model)

W_k = nn.Linear(d_model, d_model)

W_v = nn.Linear(d_model, d_model)

In [315]:
Q = W_q(x)  # (batch_size, seq_len, d_model)
K = W_k(x)  # (batch_size, seq_len, d_model)
V = W_v(x)  # (batch_size, seq_len, d_model)

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

Q shape: torch.Size([2, 5, 512])
K shape: torch.Size([2, 5, 512])
V shape: torch.Size([2, 5, 512])


In [316]:
scores = torch.matmul(Q, K.transpose(-2, -1))

print("Scores shape:", scores.shape)

Scores shape: torch.Size([2, 5, 5])


In [317]:
d_k = Q.size(-1)

scaled_scores = scores / math.sqrt(d_k)

print('d_k:', d_k)
print("Scaled Scores shape:", scaled_scores.shape)

d_k: 512
Scaled Scores shape: torch.Size([2, 5, 5])


In [318]:
attention_weights = F.softmax(scaled_scores, dim=-1)
print("Attention Weights shape:", attention_weights.shape)

Attention Weights shape: torch.Size([2, 5, 5])


In [319]:
print(
    attention_weights[0, 0]
)

print(
    attention_weights[0, 0].sum()
)

tensor([0.1980, 0.2062, 0.2171, 0.1816, 0.1971], grad_fn=<SelectBackward0>)
tensor(1., grad_fn=<SumBackward0>)


In [320]:
attention_weights = torch.matmul(attention_weights, V)

print("Output shape:", attention_weights.shape)

Output shape: torch.Size([2, 5, 512])


In [321]:
def scaled_dot_product_attention(Q, K, V, mask=None):

    d_k = Q.size(-1)

    scores = torch.matmul(Q, K.transpose(-2, -1))

    scores = scores / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attention_weights = F.softmax(scores, dim=-1)

    output = torch.matmul(attention_weights, V)

    return output

In [322]:
output, weights = scaled_dot_product_attention(
    Q,
    K,
    V
)

print("Output:", output.shape)
print("Weights:", weights.shape)

Output: torch.Size([5, 512])
Weights: torch.Size([5, 512])


In [323]:
import torch
import torch.nn as nn
import math


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # Final projection
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        """
        x:
        [batch_size, seq_len, d_model]

        returns:
        [batch_size, num_heads, seq_len, head_dim]
        """

        batch_size, seq_len, _ = x.shape

        x = x.view(batch_size,seq_len,self.num_heads,self.head_dim)

        # [B, S, H, D] -> [B, H, S, D]
        x = x.transpose(1, 2)

        return x

    def forward(self, query, key, value, mask=None):
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        # Shapes:
        # [B, H, S, head_dim]


        scores = torch.matmul(Q, K.transpose(-2, -1))

        scores = scores / math.sqrt(self.head_dim)


        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))


        attention_weights = torch.softmax(scores, dim=-1)

        attention_output = torch.matmul(attention_weights,V)

        # [B, H, S, head_dim]

        attention_output = attention_output.transpose(1, 2)

        # [B, S, H, head_dim]

        batch_size, seq_len, _, _ = attention_output.shape

        attention_output = attention_output.contiguous().view(batch_size,seq_len,self.d_model)

        output = self.W_o(attention_output)

        return output, attention_weights

In [324]:
mha = MultiHeadAttention(
    d_model=512,
    num_heads=8
)

test_input = torch.randn(2, 5, 512)

output, weights = mha(
    test_input,
    test_input,
    test_input
)

print("Input:", test_input.shape)
print("Output:", output.shape)
print("Attention:", weights.shape)

Input: torch.Size([2, 5, 512])
Output: torch.Size([2, 5, 512])
Attention: torch.Size([2, 8, 5, 5])


In [325]:
class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=5000
    ):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0,max_len,dtype=torch.float32).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0,d_model,2).float()* (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)

        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe",pe)

    def forward(self, x):

        seq_len = x.size(1)

        return (x +self.pe[:, :seq_len])

In [326]:
positional_encoding = PositionalEncoding(d_model=512)

x = torch.randn(2,10,512)

x_with_position = positional_encoding(x)

print("Before:",x.shape)

print("After:",x_with_position.shape)

Before: torch.Size([2, 10, 512])
After: torch.Size([2, 10, 512])


In [327]:
class FeedForwardNetwork(nn.Module):

    def __init__(self,d_model= 512,d_ff = 2048):
        super().__init__()

        self.linear1 = nn.Linear(d_model, d_ff)

        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):

        x = self.linear1(x)

        x = F.relu(x)

        x = self.linear2(x)

        return x

In [328]:
class EncoderLayer(nn.Module):

    def __init__(self,d_model=512,num_heads=8,d_ff=2048,dropout=0.1):
        super().__init__()

        self.mha = MultiHeadAttention(d_model,num_heads)

        self.ffn = FeedForwardNetwork(d_model,d_ff)

        self.norm1 = nn.LayerNorm(d_model)

        self.norm2 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)

        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_output, _ = self.mha(x,x,x,mask)

        x = x + self.dropout1(attn_output)

        x = self.norm1(x)

        ffn_output = self.ffn(x)

        x = x + self.dropout2(ffn_output)

        x = self.norm2(x)

        return x

In [329]:
encoder_layer = EncoderLayer()

x = torch.randn(
    2,
    10,
    512
)

output = encoder_layer(x)

print(
    "Input:",
    x.shape
)

print(
    "Output:",
    output.shape
)

Input: torch.Size([2, 10, 512])
Output: torch.Size([2, 10, 512])


In [330]:
def create_causal_mask(seq_len,device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device)).unsqueeze(0).unsqueeze(0)
    return mask

In [331]:
mask = create_causal_mask(
    5,
    device="cpu"
)

print(mask[0, 0])

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])


In [332]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model,max_seq_len=5000):
        super().__init__()

        pe = torch.zeros(max_seq_len, d_model)

        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # Shape: [1, max_seq_len, d_model]

        self.register_buffer('pe', pe)

    def forward(self, x):
        
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]

In [333]:
class FeedForward(nn.Module):

    def __init__(self, d_model=512, d_ff=2048):
        super().__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):

        x = self.fc1(x)

        x = self.relu(x)

        x = self.fc2(x)

        return x

In [334]:
class EncoderLayer(nn.Module):

    def __init__(self,d_model=512,num_heads=8,d_ff=2048,dropout=0.1):
        super().__init__()

        self.self_attention = MultiHeadAttention(d_model=d_model,num_heads=num_heads)

        self.feed_forward = FeedForward(d_model=d_model,d_ff=d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):

        # Self-attention
        attention_output, _ = self.self_attention(x, x, x, mask)

        # Residual + LayerNorm
        x = self.norm1(x + self.dropout1(attention_output))

        # Feed-forward
        ff_output = self.feed_forward(x)

        # Residual + LayerNorm
        x = self.norm2(x + self.dropout2(ff_output))

        return x

In [335]:
class Encoder(nn.Module):

    def __init__(self,num_layers=6,d_model=512,num_heads=8,d_ff=2048,dropout=0.1):
        super().__init__()

        self.layers = nn.ModuleList([
            EncoderLayer(
                d_model=d_model,
                num_heads=num_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):

        for layer in self.layers:
            x = layer(x, mask)

        return x

In [336]:
def create_causal_mask(seq_len, device):

    mask = torch.tril(torch.ones(seq_len,seq_len,device=device))

    return mask.unsqueeze(0).unsqueeze(0)

In [337]:
class DecoderLayer(nn.Module):
    def __init__(self,d_model=512,num_heads=8,d_ff=2048,dropout=0.1):
        super().__init__()

        self.self_attention = MultiHeadAttention(d_model=d_model,num_heads=num_heads)

        self.cross_attention = MultiHeadAttention(d_model=d_model,num_heads=num_heads)

        self.feed_forward = FeedForward(d_model=d_model,d_ff=d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_output, tgt_mask=None, memory_mask=None):

        # Self-attention
        self_attention_output, _ = self.self_attention(x, x, x, tgt_mask)

        # Residual + LayerNorm
        x = self.norm1(x + self.dropout1(self_attention_output))

        # Cross-attention
        cross_attention_output, _ = self.cross_attention(x, enc_output, enc_output, memory_mask)

        # Residual + LayerNorm
        x = self.norm2(x + self.dropout2(cross_attention_output))

        # Feed-forward
        ff_output = self.feed_forward(x)

        # Residual + LayerNorm
        x = self.norm3(x + self.dropout3(ff_output))

        return x

In [338]:
class Decoder(nn.Module):
    def __init__(self,num_layers=6,d_model=512,num_heads=8,d_ff=2048,dropout=0.1):
        super().__init__()

        self.layers = nn.ModuleList([
            DecoderLayer(
                d_model=d_model,
                num_heads=num_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

    def forward(self, x, enc_output, tgt_mask=None, memory_mask=None):

        for layer in self.layers:
            x = layer(x, enc_output, tgt_mask, memory_mask)

        return x